In [0]:
dbutils.widgets.text(
    "start_date",
    "2024-01-01",
    "Start Date"
)

dbutils.widgets.dropdown(
    "category_filter",
    "All",
    ["All", "Electronics", "Books", "Clothing"],
    "Category"
)

In [0]:
# COMMAND ----------

# EXERCISE: Read the widget values into variables
# YOUR CODE HERE
# start_date = dbutils.widgets.get("start_date")
# category_filter = dbutils.widgets.get("category_filter")

In [0]:
start_date = dbutils.widgets.get("start_date")
category_filter = dbutils.widgets.get("category_filter")

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## Part 2: Build an ETL Pipeline
# MAGIC
# MAGIC EXERCISE: Implement Extract, Transform, Load steps.

# COMMAND ----------

from pyspark.sql import functions as F

# EXTRACT: Raw order data
raw_orders = spark.createDataFrame(
    [
        ("2024-01-01", "O001", "Electronics", "Laptop", 999.99, 1, "completed"),
        ("2024-01-01", "O002", "Books", "Python Guide", 49.99, 2, "completed"),
        ("2024-01-02", "O003", "Electronics", "Phone", 699.99, 1, "cancelled"),
        ("2024-01-02", "O004", "Clothing", "Jacket", 129.99, 3, "completed"),
        ("2024-01-03", "O005", "Books", "ML Handbook", 69.99, 1, "completed"),
        ("2024-01-03", "O006", "Electronics", "Tablet", 449.99, 2, "completed"),
        ("2024-01-04", "O007", "Clothing", "Shoes", 89.99, 2, "completed"),
        ("2024-01-04", "O008", "Electronics", "Earbuds", 79.99, 5, "completed"),
        ("2024-01-05", "O009", "Books", "Data Science", 59.99, 3, "pending"),
        ("2024-01-05", "O010", "Clothing", "Hat", 29.99, 4, "completed"),
    ],
    ["order_date", "order_id", "category", "product", "price", "quantity", "status"],
)

print(f"Extracted {raw_orders.count()} raw orders")

Extracted 10 raw orders


In [0]:
# EXERCISE: TRANSFORM the data
# 1. Filter to only "completed" orders
# 2. Filter by start_date (order_date >= start_date)
# 3. Filter by category_filter (if not "All")
# 4. Add a "revenue" column (price * quantity)
# 5. Add a "processed_at" timestamp column
# YOUR CODE HERE

In [0]:
transformed_orders = raw_orders.filter(F.col("status") == "completed")

In [0]:
transformed_orders = transformed_orders.filter(F.col("order_date") >= start_date)

In [0]:
if category_filter != "All":
    transformed_orders = transformed_orders.filter(
        F.col("category") == category_filter
    )

In [0]:
transformed_orders = transformed_orders.withColumn(
    "revenue", 
    F.col("price") * F.col("quantity")
)

In [0]:
transformed_orders = transformed_orders.withColumn(
    "processed_at",
    F.current_timestamp()
)

In [0]:
display(transformed_orders)

order_date,order_id,category,product,price,quantity,status,revenue,processed_at
2024-01-03,O005,Books,ML Handbook,69.99,1,completed,69.99,2026-09-01T18:35:22.825Z
2024-01-03,O006,Electronics,Tablet,449.99,2,completed,899.98,2026-09-01T18:35:22.825Z
2024-01-04,O007,Clothing,Shoes,89.99,2,completed,179.98,2026-09-01T18:35:22.825Z
2024-01-04,O008,Electronics,Earbuds,79.99,5,completed,399.95,2026-09-01T18:35:22.825Z
2024-01-05,O010,Clothing,Hat,29.99,4,completed,119.96,2026-09-01T18:35:22.825Z


In [0]:
# COMMAND ----------

# EXERCISE: LOAD the transformed data into a Delta table "lab_orders_gold"
# Use overwrite mode
# YOUR CODE HERE

In [0]:
transformed_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lab_orders_gold")

In [0]:
spark.table("lab_orders_gold").show()

+----------+--------+-----------+-----------+------+--------+---------+-------+--------------------+
|order_date|order_id|   category|    product| price|quantity|   status|revenue|        processed_at|
+----------+--------+-----------+-----------+------+--------+---------+-------+--------------------+
|2024-01-03|    O005|      Books|ML Handbook| 69.99|       1|completed|  69.99|2026-09-01 18:35:...|
|2024-01-03|    O006|Electronics|     Tablet|449.99|       2|completed| 899.98|2026-09-01 18:35:...|
|2024-01-04|    O007|   Clothing|      Shoes| 89.99|       2|completed| 179.98|2026-09-01 18:35:...|
|2024-01-04|    O008|Electronics|    Earbuds| 79.99|       5|completed| 399.95|2026-09-01 18:35:...|
|2024-01-05|    O010|   Clothing|        Hat| 29.99|       4|completed| 119.96|2026-09-01 18:35:...|
+----------+--------+-----------+-----------+------+--------+---------+-------+--------------------+



In [0]:
# MAGIC %md
# MAGIC ## Part 3: Dashboard Queries
# MAGIC
# MAGIC EXERCISE: Write queries that could power a dashboard.

# COMMAND ----------

# MAGIC %sql
# MAGIC -- EXERCISE: Revenue by category (for a bar chart)
# MAGIC -- Columns: category, total_revenue, order_count
# MAGIC -- YOUR CODE HERE

In [0]:
%sql
SELECT
    category,
    SUM(revenue) AS total_revenue,
    COUNT(*) AS order_count
FROM lab_orders_gold
GROUP BY category
ORDER BY total_revenue DESC;

category,total_revenue,order_count
Electronics,1299.93,2
Clothing,299.94,2
Books,69.99,1


In [0]:
# MAGIC %sql
# MAGIC -- EXERCISE: Daily revenue trend (for a line chart)
# MAGIC -- Columns: order_date, daily_revenue, cumulative_revenue
# MAGIC -- YOUR CODE HERE (Hint: Use SUM() OVER(ORDER BY order_date) for cumulative)

In [0]:
%sql
SELECT
    order_date,
    SUM(revenue) AS daily_revenue,
    SUM(SUM(revenue)) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ----
    ) AS cumulative_revenue
FROM lab_orders_gold
GROUP BY order_date
ORDER BY order_date;

order_date,daily_revenue,cumulative_revenue
2024-01-03,969.97,969.97
2024-01-04,579.93,1549.9
2024-01-05,119.96,1669.8600000000001


In [0]:
# COMMAND ----------

# MAGIC %sql
# MAGIC -- EXERCISE: Top products by revenue (for a table/leaderboard)
# MAGIC -- Columns: product, category, total_revenue, total_quantity
# MAGIC -- Order by total_revenue DESC, limit 5
# MAGIC -- YOUR CODE HERE


In [0]:
%sql
SELECT
    product,
    category,
    SUM(revenue) AS total_revenue,
    SUM(quantity) AS total_quantity
FROM lab_orders_gold
GROUP BY product, category
ORDER BY total_revenue DESC
LIMIT 5;

product,category,total_revenue,total_quantity
Tablet,Electronics,899.98,2
Earbuds,Electronics,399.95,5
Shoes,Clothing,179.98,2
Hat,Clothing,119.96,4
ML Handbook,Books,69.99,1


In [0]:
# MAGIC %md
# MAGIC ## Part 4: Job Configuration (Conceptual)
# MAGIC
# MAGIC EXERCISE: Answer these questions about scheduling this notebook as a job.

# COMMAND ----------

# MAGIC %md
# MAGIC **Q1:** What cluster type should you use for a scheduled daily job?

#A: I would use a job cluster, since it's used for scheduled jobs because it terminates after the job completes (more cost efficient)

# MAGIC **Q2:** If this job fails, what retry configuration would you set?

#A: I'd configure automatic retries, 2 or 3, with a deley of 5 minutes between them. 

# MAGIC **Q3:** How would you pass the start_date parameter when running as a job?

#A: I'd add start_date as a parameter in the job configuration (ex. start_date = 2026-04-01). That is read by using dbutils.widgets.get("start_date")
